In [1]:
from langchain_ollama import ChatOllama
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.chat_history import (
    InMemoryChatMessageHistory,
    BaseChatMessageHistory,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    CSVLoader,
    WebBaseLoader,
    DirectoryLoader,
)
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
    BM25Retriever,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

from langchain_cohere import CohereRerank
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter, DocumentCompressorPipeline

c:\source\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\soldesk\AppData\Local\Temp\ipykernel_10808\2827418599.py:28: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# .env 내용 가져오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
    temperature=0
)

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params = {
        "temperature" : 0
    }
)

qwen_llm = ChatOllama(model="qwen3.5:4b", temperature=0)
exaone_llm = ChatOllama(model="exaone3.5:2.4b", temperature=0)

### RAG 심화




In [3]:
# pdf => chunks 반환 함수

def create_chunks_from_pdf(pdf_path, chunk_size=500, chunk_overlap=50):
    loader = PyPDFLoader(pdf_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(loader.load())

    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]

    return chunks

def create_vectorstore(chunks, embeddings, collection_name, persis_directory='./db/chroma_db'):
    db = Chroma(
        embedding_function=embeddings,
        persist_directory=persis_directory,
        collection_name=collection_name,
    )
    
    db.delete_collection()

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persis_directory,
        collection_name=collection_name,
    )


def create_retriever(
    vectorstore, search_type="similarity", k=3, fetch_k=20, lambda_mult=0.5
):
    kwargs = {"k":k}

    if search_type=="mmr":
        kwargs['fetch_k'] = fetch_k
        kwargs['lambda_mult'] = lambda_mult

    return vectorstore.as_retriever(search_type=search_type, search_kwargs=kwargs)


def print_retrieved_docs(title, retriever, query):
    docs = retriever.invoke(query)

    print("\n" + "="*50)
    print(title)
    print("\n" + "="*50)    

    for i, doc in enumerate(docs):
        print(f"\n [chunk {i}]")
        print(doc.page_content)
        print(f"\nPage : {doc.metadata.get("page")}")

In [4]:
chunks1 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 1000, 100)
chunks2 = create_chunks_from_pdf("./data/Summary of ChatGPTGPT-4 Research.pdf", 300, 30)

print(f"분할된 청크 수 : {len(chunks1)}")
print(f"분할된 청크 수 : {len(chunks2)}")

분할된 청크 수 : 118
분할된 청크 수 : 378


In [ ]:
vectorstore1 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama3")
vectorstore2 = create_vectorstore(chunks1, ollama_embedding, collection_name="gpt_research_ollama4")

ollama1_retriever = create_retriever(vectorstore1)
ollama2_retriever = create_retriever(vectorstore2)

query = 'where can i use chatGPT?'

print_retrieved_docs('ollama embedding', ollama1_retriever, query)
print_retrieved_docs('ollama embedding', ollama2_retriever, query)

### 2. MMR (Maximal Marginal Relevance) Retriever
 - 관련성(Relevace)과 다양성(Diversity) 고려
 - 법률 문서, 기술 매뉴얼 유사 내용이 반복되는 문서에서 효과적이다.

In [ ]:
chunks1 = create_chunks_from_pdf(
    "./data/2026 상 삼성전자 DX부문 직무기술서.pdf",
    1000,
    100,
)

vectorstore1 = create_vectorstore(
    chunks1, ollama_embedding, collection_name="samsung_watson1", persist_directory="./db/samsung_watson_db"
)

mmr_retriever = create_retriever(vectorstore1, search_type="mmr", k=5)
similarity_retriever = create_retriever(vectorstore1, k=5)

query = '마케팅 - 제품/서비스 마케팅 포지션은?'

print_retrieved_docs('MMR', mmr_retriever, query)
print_retrieved_docs('similarity', similarity_retriever, query)


TypeError: create_vectorstore() got an unexpected keyword argument 'persist_directory'

# 3. selfquery retriever
- 자연어 질문을 분석하여 시멘틱 검색쿼리와 메타데티어 필터를 llm이 자동 생성하게 하는 고급 retriever
- 질문 : 2023 년 이후 기약금액이 1억 이상인 계약 찾아줘 => LLM
    시멘틱 검색 쿼리 : 계약
    filter : {year >= 2023, 계약금액 >= 100000}

In [ ]:
docs = [
    Document(
        page_content="삼성전자 제품 마케팅 직무입니다.",
        metadata={
            "year":2025,
            "department":"marketing"
        }
    ),
        Document(
        page_content="삼성전자 회계 직무 입니다.",
        metadata={
            "year":2024,
            "department":"accounting"
        }
    ),
        Document(
        page_content="삼성전자 백엔드 직무입니다.",
        metadata={
            "year":2023,
            "department":"backend"
        }
    )
]

metadata_field_info = [
    AttributeInfo(
        name="year", description="문서작성 연도", type="integer"),
    AttributeInfo(
        name="department", description="직무 부서", type="string"
    )
]


document_content_description = "회사 내부 문서 및 직무 자료"

In [ ]:
vectorstore1 = create_vectorstore(
    docs,
    watson_embedding,
    collection_name="selfquery",
    persis_directory='./db/watson_chroma'
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    metadata_field_info=metadata_field_info,
    document_contents=document_content_description,
    verbose=True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator()
)


In [ ]:
question = "2023년 이후 backend 부서 직무를 찾아줘"
self_query_retriever.invoke(question)

[Document(id='734cacf4-b2c3-4d22-b314-72e93ebf0e09', metadata={'year': 2023, 'department': 'backend'}, page_content='삼성전자 백엔드 직무입니다.'),
 Document(id='07da8043-f0f8-442c-acb8-218c40a40b45', metadata={'year': 2023, 'department': 'backend'}, page_content='삼성전자 백엔드 직무입니다.')]

In [ ]:
docs = [
    Document(
        page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳 까지 수분을 공급합니다.",
        metadata={"year": 2024, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
        metadata={"year": 2023, "category": "메이크업", "user_rating": 3},
    ),
    Document(
        page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
        metadata={"year": 2023, "category": "스킨케어", "user_rating": 4},
    ),
    Document(
        page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA+++ 높은 자외선 차단지수로 피부를 보호합니다.",
        metadata={"year": 2025, "category": "선케어", "user_rating": 5},
    ),
    Document(
        page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
        metadata={"year": 2024, "category": "메이크업", "user_rating": 4},
    ),
]

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(name="year", description="화장품 출시 연도", type="integer"),
    AttributeInfo(
        name="category",
        description="화장품 카테고리 ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="user_rating",
        description="화장품 평점 (1~5)",
        type="integer",
    ),
]

document_content_description = "화장품 제품 정보"

vectorstore1= create_vectorstore(
    docs,
    watson_embedding,
    collection_name="selfquery2",
    persis_directory="./db/watson_chroma"
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=watson_llm,
    vectorstore=vectorstore1,
    metadata_field_info=metadata_field_info,
    document_contents=document_content_description,
    verbose=True,
    enable_limit=True,
    structured_query_translator=ChromaTranslator(),
)

In [ ]:
self_query_retriever.invoke("2024년 이후로 평점이 4 이상인 제품을 추천해줘")

[Document(id='8934e028-f8bc-4867-8dbf-964d1583154d', metadata={'category': '메이크업', 'user_rating': 4, 'year': 2024}, page_content='롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.'),
 Document(id='f4f3e526-042f-45e8-96be-68b72d94a8dc', metadata={'user_rating': 5, 'year': 2025, 'category': '선케어'}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA+++ 높은 자외선 차단지수로 피부를 보호합니다.'),
 Document(id='68948598-a9e3-4680-9feb-1ee48e525d10', metadata={'year': 2024, 'category': '스킨케어', 'user_rating': 4}, page_content='수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳 까지 수분을 공급합니다.')]

In [ ]:
self_query_retriever.invoke("카테고리가 선케어인 상품을 추천해줘")

[Document(id='f4f3e526-042f-45e8-96be-68b72d94a8dc', metadata={'category': '선케어', 'year': 2025, 'user_rating': 5}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA+++ 높은 자외선 차단지수로 피부를 보호합니다.')]

### OCR

In [ ]:
pdf_path = "./data/2026 상 삼성전자 DX부문 직무기술서.pdf"
chunks = create_chunks_from_pdf(pdf_path)
for i in range(2):
    print("="*50)
    print(chunks[i].page_content[:500])

회로개발
회로 기술을 기반으로 삼성전자 제품 및 솔루션의 혁신적인 가치를 창출합니다
•
•
•
•
•
•
•
•
•
•
•
•
•
•


In [ ]:
# PDF를 이미지로 변환하기
import os

import fitz

output_path = "./output"
os.makedirs(output_path, exist_ok=True)

pdf = fitz.open(pdf_path)

for page_num in range(len(pdf)):
    page = pdf[page_num]
    pix = page.get_pixmap(dpi=300)
    pix.save(os.path.join(output_path, f"page_{page_num}.png"))

pdf.close()

In [ ]:
import easyocr

reader = easyocr.Reader(["ko", "en"])
result = reader.readtext("./output/page_19.png", detail=0, paragraph=True)
print("\n".join(result))

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


DA사업부(수원 근무) 해외영업 B2C 영업
포지션 소개 Job Overview
글로벌 시장과 소비자에 대한 이해틀 바탕으로 생활가전 제품의 판매 전락올 수립하고 매출 확대와 브랜드 리더십올 제고합니다:
수행업무 Job Details
담당 지역의 시장과 고객 특성, 판매 데이터들 분석하여 매출과 손의 극대화틀 위한 제품 가격유통 마켓팅올 아우르는 장단기 판매 전락올 수립합니다. 수립든 판매 전락올 기반으로 담당 법인과 현업하여 적기 공급올 위한 오퍼레이선올 지원하고 현지 법인의 실행 계획과 진척올 관리 및 개선합니다.
자격요건 Requirements
영어로 해외 자료 조사 및 커류니키이선이 가능하신 분 Global 이문화에 대한 이해도가 높으신 분 팀위크와 협업 능력올 보유하신 분
우대사항 Preferences
직무와 연관된 대내외 활동 경험올 보유하신 분 제2외국어 회화 역량울 보유하신 분
커리어 비전 Career Vision
담당 지역의 언어와 문화 습득올 포함한 글로벌 역량울 강화할 수 있으려 고객시장 유통 특성 등 전문 지식 습득과 법인 관리 및 개선올 통한 영업 실무 경험올 쌍울 수 있습니다. 삼성전자 해외영업 주재원으로서 현지 법인에서 근무하여, 매출 확대와 브랜드 리더십올 공고히 하는 현장 경험올 익히려 글로벌 영업 전문가로 성장할 수 있습니다:
'글로벌올 무대로 DA의 새로운 가능성울 열어갈 당신에게"
시장과 소비자에 대한 이해틀 바탕으로
생활가전 브랜드틀 최고의 자리로 이끌어 칼 인재틀 모십니다.


In [ ]:
import json

pages = []

for page_num in range(31):
    image_path = f"./output/page_{page_num}.png"
    result = reader.readtext(image_path, detail=0, paragraph=True)
    text = "\n".join(result)
    pages.append({"page": page_num+1, "content": text})

with open("./data/samsung_dx_ocr.json", "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' a

### BM25

- 유사도 검색은 유사성 위주 (정확한 제품명, 코드, 버전, 고유명사 등의 키워드 매칭에는 약함)
- 유사도 검색의 단점 보완 (semantic + sparse)
- 예
    - 환불 가능한 기간이 어떻게 되나요? 유사도 검색 유리
    - ERR_CONNECTION 오류 -> 키워드 검색

In [ ]:
with open("./data/samsung_dx_ocr.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

docs = []

for page in pages:
    docs.append(
        Document(
            page_content=page["content"],
            metadata={
                "page": page["page"],
                "source": "samsung_dx_ocr",
                
                },
        )
    )

In [ ]:
# len (docs)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)

vectorstore = create_vectorstore(
    chunks=chunks,
    embeddings=watson_embedding,
    persis_directory="./db/chroma_db",
    collection_name="samsung_dx"
)

In [ ]:
# 유사도 검색
dense_retriever = create_retriever(vectorstore)

# 키워드 검색(법령, 제품명...)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

query = "시스템 소프트웨어 자격 요건에 운영체제 개념도 포함되어 있는가?"

print_retrieved_docs("hybrid", hybrid_retriever, query)


hybrid


 [chunk 0]
자격요건 Requirements
컴퓨터 전기 전자, 기계 로봇공학 등 관련 전공올 하신 분 운영체제 기본 개념에 대한 이해도틀 보유하신 분 요구사항울 분석하여 소프트웨어들 구조적으로 설계 및 구현하는 역량울 보유하신 분 다양한 분야의 엔지니어와 적극적으로 소통하여 협업할 수 짓는 역량울 보유하신 분
우대사항 Preferences
CIC++ 기반 시스템 프로그래망 역량울 보유하신 분 Linux 기반 임베디드 시스템 개발 경험올 보유하신 분(Kernel, Device Driver, BSP 등) CAN EtherCAT, UDP 등 하드웨어 통신 프로토록 활용 경험 보유하신 분 ROS2 기반 로보텍스 프로적트 수행 경험올 보유하신 분 Git 기반 협업 및 CICD 환경에서의 개발 경험올 보유하신 분
커리어 비전 Career Vision

Page : 7

 [chunk 1]
미래로봇추진단(서울 근무) S/W개발 시스템 소프트웨어
포지션 소개 Job Overview
휴머노이드 로봇의 실시간 제어 시스템 및 소프트웨어 플렉품올 개발하는 직무입니다: 운영체제 환경 구성, 디바이스 드라이버, 실시간 제어 프레임위크 등 로봇 동작의 핵심 기반이 되는 소프트웨어름 설계 및 개발하다, 하드웨어 설계 조직 및 Al 연구 조직과 긴밀히 현업합니다:
수행업무 Job Details
휴머노이드 로봇의 실시간 제어 프레임위크 설계하고 개발합니다: 모터; 센서 등 로봇 하드웨어 제어름 위한 디바이스 드라이버 및 하드웨어 추상화 계층올 개발합니다. 로봇 제어용 운영체제(Linux 기반 실시간 OS 등) 환경올 구성하고 시스템 성능올 최적화합니다: 유관 부서와 협업하여 보행 조작 등 Al 기능과 제어 시스템 간 연동 미들웨어름 개발합니다:
자격요건 Requirements

Page : 7

 [chunk 2]
S/W개발 소프트웨어 기술에 대한 전문적인 지식올 기반으로 창의적이고 분석적인 사고름 통해 신기술올 선도하고 당사 제품에 반영함으로써 제품 및 슬루선의

### ReRanking & Contextual Compression
- 유사도 검색 (Bi-Encoder) : 쿼리(질문)와 문서를 각각 벡터로 변환 후 계산
- Cross-Encoder : 쿼리+문서 쌍으로 벡터로 변환
    - 정확도가 매우 높으나, 매우 느림

In [ ]:
!pip install cohere langchain-cohere sentence-transformers

In [ ]:
# 기본검색(유사도 검색)
base_retriever = dense_retriever = create_retriever(vectorstore, k=20)
query = "서울 근무 직무는 어떤 항목에 지원해야하나요?"
docs = base_retriever.invoke(query)

# 20개 후보군을 대상으로 reranking
reranker = CohereRerank(model="rerank-v4.0-pro", top_n=5)

compression_retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=base_retriever)

# 최종
print_retrieved_docs("Rerank", compression_retriever, query)


Rerank


 [chunk 0]
1. 유치지원: 국제회의를 제주로 유치하는 단계에서 지원
기준 지원내용
참가자 100명이상, 해외참가자
3개국 50명 이상, 2일 이상
· 전차대회(유관대회) 홍보부스 제작 및 운영비
·  유치제안서 및 온오프라인 홍보물(PT, 인쇄물, 영상, 
기념품 등)  
·  해외공식연회(KoreanNight·Lunch)
·  회의 유치 직접적인 활동 참가자 항공 (2인 이내)
2. 홍보지원: 참가자 증대를 위한 사전 홍보 단계에서 지원
기준 지원내용
참가자 100명이상, 해외참가자
3개국 50명 이상, 2일 이상
· 전차대회(유관대회) 홍보부스 제작 및 운영비
· 온오프라인 홍보물(인쇄물, 배너, 영상, 기념품 등)
3. 개최지원: 회의가 제주에서 개최되는 단계에서 지원
- 국내외 회의 개최지원
구  분 선정기준 개최일 지원내용
국제회의
해외참가자 500명 이상
2일이상
· 행사장 임대료
· 기념품 제작비
· 오만찬비
· 셔틀버스 운영비
회의규모 100명 이상
(해외참가자 50명 이상)

Page : 36

 [chunk 1]
카카오톡�플러스친구 
[제주관광공사]검색하여�채팅�상담�가능

Page : 38

 [chunk 2]
여실객실보유
· 최대 4,300명 수용 가능한 제주국제컨벤션센터 및 다양한 컨벤션 시설을 갖춘 호텔&
리조트 보유
· 제주의독특한문화를담은유니크베뉴및 다양한 팀빌딩프로그램운영
검증된국제회의도시
· 풍부한국제회의경험 : 다수의국가간정상회의, 국빈급국제회의성공개최노하우축적
· 2015년아시아최고의국제회의개최지선정 :유럽비즈니스여행전문지
「BusinessDestination」선정)
· 대한민국 최고의 만족도 국제회의개최도시 : 제주 4.42/5점만점(2017년한국관광공사  
시행‘2017MICE참가자조사’보고서)
세계의보물섬,유네스코가인정한자연의도시
· 세계7대재연경관선정(‘11.11.11)
· 전세계 유일의 유네스코 자연과학 3개분야 인증지역 : 생물권보전지역(’02.12월),세계
자연유산(‘07.6월),세계지

### LLMChainExtractor

- 질문+문서를 같이 LLM에게 보내서 질문과 관련된 내용만 추출
- 현재 chunks 안에 있는 내용을 줄여내는 것

In [ ]:
# 문서로드 / 청크 추출
chunks = create_chunks_from_pdf("./data/제주관광가이드.pdf")

# 인덱싱 - vectorstore
vectorstore = create_vectorstore(chunks, watson_embedding, collection_name="jeju_guide")

# 질의
base_retriever = create_retriever(vectorstore=vectorstore, k=20)

docs = base_retriever.invoke("생활 속 제주어 에서 엄불랑하다는 무슨 뜻이야?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

생활 속 제주어
제주는 타 지역보다 한국어의 고형(古形)을 많이 유지하고 있는 동시에 
제주도만의 고유한 어휘나 문법적 특성을 가지고 있다.
다른 지역 사람이 못 알아듣는 제주어
제주어 뜻풀이
솔쩨기 살짝
안네다 드리다
베지근허다 입안에 기름기가 감돌아 맛이 있다.
엄불랑허다 어마어마하다
코시롱허다 고소하다
산도록허다 시원하다  예) 물이 산도록헌 게 좋다.
두령청이 우두망찰
무사 왜
영, 경, 정 이렇게, 그렇게, 저렇게
게메 글쎄
인사말
제주어 뜻풀이
펜안허우꽈? 편안(안녕)하십니까?
제주도 오난 어떵허우꽈? 제주도에 오니 어떠십니까?
차말로 좋수다. 참말로 좋습니다.
공기도 마고, 산이영 바다잉여 마딱 좋은게마씀 공기도 맑고, 산이랑 바다랑 모두 좋네요.
서울 갈 때랑 하영 다앙 갑서. 서울 갈 때는 많이 담아서 가십시오.
게메양. 경 헤시민 얼마나 좋코마씀? 글쎄요. 그렇게 했으면 얼마나 좋겠습니까?
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
----------
식당에서
제주어 뜻풀이
무신걸 먹으코? 무엇을 먹을까?
게메양, 제주도에만 이신 거 먹게마씀. 글쎄요, 제주도에만 있는 것 먹게요.
구젱기영 무꾸럭이영 오토미영 하간 거 다 잇수다. 소라랑 문어랑 옥돔이랑 온갖 거 다 있습니다.
모멀펌벅에 자리젯! 메밀범벅에 자리젓!
먹을 것도 잘도 하신게. 먹을 것도 정말(엄청) 많네.
타 지역과 의미가 다른 제주어
제주어 뜻풀이
감저 고구마
지슬, 지실 감자
-허게 하자(청유의 의미) 공부허게: 공부하자
글라 가자 예) 장에 글라.(장에 가자.)
호미 낫
가겡이, 가각지, 가게 호미
폭삭 속앗수다 무척 수고하셨습니다.
삼춘 ①삼촌    ②나이 든 남자 어른이나 여자 어른을 부르는 말.
요망지다 야무지다
아 까다 아깝다(사랑스럽고 귀엽다)
젊은 층도 자주 사용하는 제주어
제주어 뜻풀이
뭐허멘? 뭐하니?
가이, 자이, 야이 걔, 쟤, 얘
기 그래
뒌 됐어
허쿠다 하겠습니다
잘도 정말, 엄청
벨라지다 바라지다
----------
‘깅이’는 ‘게’의 제

In [ ]:
# 답변만 축약

extractor = LLMChainExtractor.from_llm(watson_llm)

compression_retriever = ContextualCompressionRetriever(base_compressor=extractor, base_retriever=base_retriever)

docs = compression_retriever.invoke("안녕하세요를 제주어로 하면 무엇인가?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

### Embedding Filter

- 임계값을 기준으로 미달한 문서 제외

In [ ]:
embedding_filter = EmbeddingsFilter(embeddings=watson_embedding, similarity_threshold=0.8)

pipeline = DocumentCompressorPipeline(transformers=[embedding_filter, extractor])

compression_retriever = ContextualCompressionRetriever(base_compressor=pipeline, base_retriever=base_retriever)

docs = compression_retriever.invoke("참말로 좋습니다를 제주어로 한다면?")

for doc in docs:
    print(doc.page_content[:500])
    print('-'*10)

In [ ]:
filtered_docs = embedding_filter.compress_documents(docs,query)

len(filtered_docs)

### RAG 성능 개선
1. 문서 전처리
- chunk 최적화, metadata 추가

2. Retrieval(검색) 개선
- MMR, BM25, Hybrid, SelfQuery 이용

3. Retrieval 후 처리
- Rerank, EmbeddingFilter, LLMChainExtractor, ContextualCompressionRetriever

In [ ]:
import easyocr

reader = easyocr.Reader(["ko", "en"])

input_pdf = "./data/한양대 대학원 캠퍼스 가이드.pdf"
path_name = "hanyang_graduate"


def convert_pdf_to_image(path_name, pdf_path):
    import os
    import fitz

    output_path = f"./output/{path_name}"
    os.makedirs(output_path, exist_ok=True)

    pdf = fitz.open(pdf_path)

    for page_num in range(len(pdf)):
        page = pdf[page_num]
        pix = page.get_pixmap(dpi=300)
        pix.save(os.path.join(output_path, f"page_{page_num}.png"))

    pdf.close()


def convert_image_to_json(path_name):
    import os
    import json

    image_dir = f"./output/{path_name}"
    num_pages = len([f for f in os.listdir(image_dir) if f.endswith(".png")])

    pages = []

    for page_num in range(num_pages):
        image_path = f"{image_dir}/page_{page_num}.png"
        result = reader.readtext(image_path, detail=0, paragraph=True)
        text = "\n".join(result)
        pages.append({"page": page_num + 1, "content": text})

    os.makedirs(f"./data/{path_name}", exist_ok=True)
    with open(f"./data/{path_name}/result.json", "w", encoding="utf-8") as f:
        json.dump(pages, f, ensure_ascii=False, indent=2)


# 이미지 기반 PDF라 PyPDFLoader로는 텍스트가 안 나옴.
# OCR 결과(result.json)를 읽어서 chunk를 만든다.
def create_chunks_from_json(path_name, chunk_size=500, chunk_overlap=50):
    import json
    from langchain.schema import Document

    with open(f"./data/{path_name}/result.json", encoding="utf-8") as f:
        pages = json.load(f)

    docs = [
        Document(page_content=p["content"], metadata={"page": p["page"]})
        for p in pages
        if p["content"].strip()
    ]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    chunks = [chunk for chunk in chunks if chunk.page_content.strip()]
    print(len(chunks))

    return chunks


def create_vectorstore(
    chunks, embeddings, collection_name, persist_directory
):

    db = Chroma(
        embedding_function=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )

    db.delete_collection()

    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name,
    )


# 1. PDF -> 이미지 -> OCR(json)
convert_pdf_to_image(path_name=path_name, pdf_path=input_pdf)
convert_image_to_json(path_name=path_name)

# 2. OCR 결과(json) -> chunk -> 벡터스토어
chunks = create_chunks_from_json(path_name)
create_vectorstore(
    chunks=chunks,
    embeddings=watson_embedding,
    collection_name=path_name,
    persist_directory=f"./db/{path_name}",
)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\source\ollama\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\

In [ ]:
hanyang_vectorstore = \
    Chroma(
        embedding_function=watson_embedding,
        persist_directory=f"./db/{path_name}",
        collection_name=path_name,
    )

### 소비자보호원 서비스집단 분쟁조정 사례집 RAG

In [ ]:
from langchain_ibm import ChatWatsonx
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
)

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)

from dotenv import load_dotenv
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_ibm import WatsonxEmbeddings
from langchain_chroma import Chroma

from pydantic import BaseModel, Field
from langchain_core.runnables import RunnablePassthrough

import re

from pprint import pprint

load_dotenv()

apikey=  os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watson_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    apikey=apikey,
    project_id=project_id,
    watsonx_url=watson_ai_url,
    model_id="ibm/granite-4-h-small",
    max_tokens=2000,
    params = {
        "temperature": 0
    }
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watson_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)

# 문서 로드
pdf_path = "./data/2018 서비스·집단 분쟁조정 사례집.pdf"

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(len(docs)) # 예상 값 : 200
pprint(docs[0].page_content)
pprint(docs[0].metadata)

# 전처리 - 정규식을 이용한 내용 추출
pattern = r"사\s*례\s*\d+.*?사건번호.*?결정일자.*?\d{4}\.\s?\d{1,2}\.\s?\d{1,2}\."

target_text = "".join(docs[10].page_content)
split_text = re.findall(pattern, target_text, re.DOTALL)

parts = []
if split_text: # 패턴이 존재할때 패턴을 기준으로 문서 분리
    parts = re.split(pattern, target_text)

    # 사례번호
    case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
    print(case_no[0] if case_no else "사례번호 미검출")
else:
    print("패턴 매칭 결과 없음")

# 사건 번호, 결정일자 추출
class CaseMetadata(BaseModel):
    case_number:str = Field(description="사건번호 예: 2018일나565")
    decision_date:str = Field(description="결정일자 예: 2018. 8. 7")


metadata_prompt = PromptTemplate.from_template(
    """\
        다음은 분쟁 조정 사례에 대한 텍스트입니다.

        - case_number : 사건번호
        - decision_date : 결정일자

        반드시 JSON 으로만 반환하세요
        {case_text}
    """
)

structed_llm = watson_llm.with_structured_output(CaseMetadata)
chain = metadata_prompt | structed_llm

if split_text:
    case_metadata = chain.invoke({
        "case_text" : split_text[0]
    })

    print(case_metadata)
    print(dict(case_metadata))


# 주 문 ~~~ (탐색용: parts 가 분할됐고 두 번째 조각이 있을 때만)
if len(parts) > 1:
    pprint(parts[1])

    m = re.search(r"주 문\n", parts[1])
    if m:
        title = parts[1][:m.span()[0]].replace("\n", "").strip()      # 주문 앞부분 = 제목
        content = parts[1][m.span()[0]:].strip()                       # 주문 이후 = 내용
        pprint(title)
        pprint(content)

# 사례 번호, 사건 번호, 결정일자, 제목은 meta데이터 추가
docs = []
case_metadata = {}

# 사건이 시작되는 페이지 ~ 마지막에서 -2 페이지 까지 반복
for doc in docs[10:-2]:
    split_text = re.findall(pattern, "".join(doc.page_content), re.DOTALL)
    if split_text:  # 새 사례가 시작되는 페이지
        case_metadata = {}

        # 사례 번호 추출
        case_no = re.findall(r"례\s?(\d+)\s?사건번호", split_text[0])
        case_metadata["case_no"] = case_no[0] if case_no else ""

        # 패턴 기준으로 텍스트 분할
        parts = re.split(pattern, "".join(doc.page_content))

        if len(parts) > 1 and re.search(r"주 문\n", parts[1]):
            span = re.search(r"주 문\n", parts[1]).span()
            # 제목 추출 (주문 앞부분)
            case_metadata["title"] = parts[1][:span[0]].replace("\n", "").strip()
            # 내용 추출 후 기존 내용 업데이트 (주문 이후)
            doc.page_content = parts[1][span[0]:].strip()
        else:
            case_metadata["title"] = ""

        # 사건 번호, 결정 일자 추출
        i = 0
        while i < 10:
            try:
                response = chain.invoke({"case_text": split_text[0]})
                for k, v in dict(response).items():
                    case_metadata[k] = v.replace("\n", "").replace(" ", "")
                break
            except Exception:
                i += 1
                continue

        doc.metadata.update(case_metadata)
        docs.append(doc)
    else:
        doc.metadata.update(case_metadata)
        docs.append(doc)

len(docs)

200
'소비자분쟁조정위원회\n2018\n서비스·집단 \n분쟁조정 사례집'
{'author': 'PC_A2',
 'creationdate': '2019-06-05T11:33:24+09:00',
 'creator': 'PScript5.dll Version 5.2.2',
 'moddate': '2019-06-05T11:58:31+09:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Acrobat Distiller 9.0.0 (Windows)',
 'source': './data/2018 서비스·집단 분쟁조정 사례집.pdf',
 'title': '<32303139303630355FBCD2BAF1C0DABFF820BBE7B7CAC1FD5BBCADBAF1BDBA5D5FB3BBC1F65FC6EDC1FDBABB76657231312E687770>',
 'total_pages': 200}
01
case_number='2018일나565' decision_date='2018. 8. 7.'
{'case_number': '2018일나565', 'decision_date': '2018. 8. 7.'}
('\n'
 '세탁 후 갑피 마모 및 경화된 가죽 \n'
 '운동화에 대한 손해배상 요구\n'
 '주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 

188

# 2. 분할

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(docs)

print(split_docs[0].page_content)

소비자분쟁조정위원회
2018
서비스·집단 
분쟁조정 사례집


In [ ]:
# 마침표 뒤에 나오는 줄바꿈 문자를 그대로 두고 나머지 줄바꿈 문자만 제거
pprint(split_docs[18].page_content)

text = split_docs[18].page_content

text = re.sub(r"(?<!\.)\n"," ", text)
pprint(text)

('주 문\n'
 '1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽 \n'
 '운동화, 색상 : 흰색) 1켤레를 반환한다. \n'
 '2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원\n'
 '을 지급한다.\n'
 '이 유\n'
 '1. 기초사실\n'
 '가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색, \n'
 '이하 ‘이 사건 제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10. \n'
 '피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 4,000원)하였는데 수령 후 갑피 \n'
 '마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였\n'
 '으나, 이후에도 경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신\n'
 '청인에게 손해배상(세탁비 환급 포함)을 요구하였으며, 피신청인은 세탁과실이 없\n'
 '다는 이유로 이를 거부하였다.\n'
 '나. 한국소비자원 신발제품심의위원회 심의 결과는 다음과 같다.')
('주 문 1. 신청인은 2018. 10. 16.까지 피신청인에게 이 사건 제품(제품명 : ○○○○ 가죽  운동화, 색상 : 흰색) 1켤레를 '
 '반환한다.  2. 피신청인은 신청인으로부터 제1항 제품을 반환받음과 동시에 신청인에게 71,000원 을 지급한다.\n'
 '이 유 1. 기초사실 가. 신청인은 2017. 6. 6. 가죽 운동화(제품명 : ○○○○ 가죽 운동화, 색상 : 흰색,  이하 ‘이 사건 '
 '제품’) 1켤레를 160,200원에 구매하여 착화하였고, 2018. 1. 10.  피신청인에게 이 사건 제품의 세탁을 의뢰(세탁비 '
 '4,000원)하였는데 수령 후 갑피  마모 및 경화된 사실(이하 ‘이 사건 현상’)을 확인하여 피신청인이 재세탁을 하였 으나, 이후에도 '
 '경화현상만 다소 개선될 뿐 갑피 마모 현상이 개선되지 않아 피신 청